# Email Authentication Security Audit of the Irish Public Sector

Analysis notebook. Run top to bottom. Section numbers match Chapter 5 of the thesis.

**5.1** Sample characteristics · **5.2** SPF · **5.3** DKIM · **5.4** DMARC ·
**5.5** Combined effectiveness · **5.6** Variation across the sector · **5.7** Attack exposure

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

df = pd.read_csv("results.csv", encoding="cp1252")
N = len(df)

# repair mojibake from the cp1252 export (e.g. An Garda Siochana)
df["organization_name"] = (df["organization_name"]
                           .str.encode("cp1252", errors="ignore")
                           .str.decode("utf-8", errors="ignore"))

print(df.shape)

In [ ]:
# ---- shared palette: used by every figure in this notebook ----------
STRONG = "#4C8577"   # meets recommendation
PARTIAL = "#E3B23C"  # partial / near limit
WEAK = "#C1666B"     # weakness
PRIMARY = "#1F4E66"  # effective / general
LIGHT = "#7F9BB5"    # published / presence
GREY = "#8A8A8A"     # not measured

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.axisbelow": True,
    "font.size": 10,
})

def counts(col):
    """Value counts with percentages, as a flat table."""
    t = df[col].value_counts(dropna=False).to_frame("domains").rename_axis(None)
    t["pct"] = (t["domains"] / N * 100).round(1)
    return t

def label_bars(ax, horizontal=False, fmt="{:.0f}"):
    """Write the value at the end of each bar."""
    for c in ax.containers:
        ax.bar_label(c, fmt=fmt, padding=3, fontsize=9)

## 5.1 Sample characteristics

Sector distribution, the collapse to five analysis groups, and sending infrastructure.

In [ ]:
print("unique domains:", df["domain"].nunique(), "of", N)
print("missing organisation names:", df["organization_name"].isna().sum())
print("failed queries:",
      df[["spf_status", "dmarc_status", "dkim_status"]].isin(["timeout", "error"]).sum().sum())

In [ ]:
counts("sector")

In [ ]:
# Health kept as its own group; six categories with n<=2 become Other Public Body
KEEP = ["Local Government", "Central Government", "Education", "Health"]
df["sector_group"] = df["sector"].where(df["sector"].isin(KEEP), "Other Public Body")

SECTOR_ORDER = ["Central Government", "Local Government", "Education",
                "Health", "Other Public Body"]
DENOM = df["sector_group"].value_counts().reindex(SECTOR_ORDER)

counts("sector_group")

**Figure 5.1: Sector distribution**

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))

s = df["sector"].value_counts().sort_values()
ax1.barh(s.index, s.values, color=[WEAK if v <= 2 else PRIMARY for v in s])
for i, v in enumerate(s):
    ax1.text(v + 0.4, i, str(v), va="center", fontsize=9)
ax1.set_xlim(0, 36)
ax1.set_xlabel("domains")
ax1.set_title("As collected (10 sectors)")

g = df["sector_group"].value_counts().reindex(SECTOR_ORDER).sort_values()
ax2.barh(g.index, g.values,
         color=[WEAK if i == "Other Public Body" else PRIMARY for i in g.index])
for i, v in enumerate(g):
    ax2.text(v + 0.4, i, f"{v}  ({v/N*100:.1f}%)", va="center", fontsize=9)
ax2.set_xlim(0, 36)
ax2.set_xlabel("domains")
ax2.set_title("As analysed (5 groups)")

plt.suptitle("Sector distribution before and after collapsing (n=78)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
def infra(s):
    s = s.lower()
    if "protection.outlook.com" in s:
        return "Microsoft 365"
    if "spf.google.com" in s:
        return "Google Workspace"
    return "Other / on-premise"

df["mail_infra"] = df["spf_record"].apply(infra)
counts("mail_infra")

**Figure 5.2: Sending infrastructure provider distribution**

In [ ]:
INFRA_ORDER = ["Microsoft 365", "Other / on-premise", "Google Workspace"]
v = df["mail_infra"].value_counts().reindex(INFRA_ORDER)[::-1]

ax = plt.subplots()[1]
ax.barh(v.index, v.values, color=PRIMARY)
for i, x in enumerate(v):
    ax.text(x + 1, i, f"{x}  ({x/N*100:.1f}%)", va="center", fontsize=9)
ax.set_xlim(0, 78)
ax.set_xlabel("domains")
ax.set_title("Sending infrastructure")
plt.tight_layout()
plt.show()

## 5.2 SPF

In [ ]:
counts("spf_status")

In [ ]:
counts("spf_qualifier")

**Figure 5.3: SPF qualifier distribution**

In [ ]:
q = df["spf_qualifier"].value_counts().reindex(["-all", "~all", "?all"])

ax = q.plot.bar(color=[STRONG, PARTIAL, WEAK], rot=0)
label_bars(ax)
ax.set_ylim(0, 62)
ax.set_ylabel("domains")
ax.set_xlabel("terminal qualifier")
ax.set_title("SPF qualifier strictness")
plt.tight_layout()
plt.show()

In [ ]:
# the single ?all record
df.loc[df["spf_qualifier"] == "?all", ["organization_name", "domain"]]

In [ ]:
# confirms the 1-4 scoring scale used in Section 4.5
pd.crosstab(df["spf_qualifier"], df["spf_qualifier_score"])

In [ ]:
df["spf_lookup_count"].describe().to_frame("lookups").round(2)

**Figure 5.4: SPF DNS-lookup distribution**

In [ ]:
lc = df["spf_lookup_count"]
counts_, edges, patches = plt.hist(lc, bins=np.arange(-0.5, 12.5), edgecolor="white")

for i, p in enumerate(patches):
    p.set_facecolor(WEAK if i >= 11 else PARTIAL if i >= 8 else PRIMARY)

plt.axvline(10.5, color=WEAK, linestyle="--", lw=1.3)
plt.xticks(range(0, 12))
plt.xlabel("DNS lookups required")
plt.ylabel("domains")
plt.title("SPF DNS lookups per domain")
plt.legend(handles=[Patch(color=PRIMARY, label="0-7 (headroom)"),
                    Patch(color=PARTIAL, label="8-10 (at limit)"),
                    Patch(color=WEAK, label=">10 (permerror)")],
           frameon=False, fontsize=8.5)
plt.tight_layout()
plt.show()

In [ ]:
print("exceeds limit:", df.loc[df["spf_lookup_exceeds_limit"], "domain"].tolist())

near = df["spf_lookup_count"].between(8, 10)
print(f"at 8-10 lookups: {near.sum()} ({near.sum()/N*100:.1f}%)")
print(df.loc[near, "sector_group"].value_counts().to_string())

In [ ]:
df["spf_effective"] = (df["spf_qualifier"].isin(["-all", "~all"])
                       & ~df["spf_lookup_exceeds_limit"])
print("SPF effective:", df["spf_effective"].sum())

## 5.3 DKIM

In [ ]:
# a failed selector probe proves only that no candidate selector resolved
df["dkim_status"] = df["dkim_status"].replace({"not_published": "undetermined"})
counts("dkim_status")

In [ ]:
undet = df.loc[df["dkim_status"] == "undetermined"]
print(undet[["organization_name", "domain", "mail_infra"]].to_string(index=False))
print()
print(pd.crosstab(df["mail_infra"], df["dkim_status"]).to_string())

In [ ]:
counts("dkim_selector")

In [ ]:
counts("dkim_key_bits")

In [ ]:
counts("dkim_key_type")

**Figure 5.5: DKIM key length by selector**

In [ ]:
ct = pd.crosstab(df["dkim_selector"], df["dkim_key_bits"])
ct = ct.loc[ct.sum(axis=1).sort_values(ascending=False).index]

ax = ct.plot.bar(stacked=True, color=[WEAK, STRONG], figsize=(8, 4.2), rot=0, width=0.7)
for i, (sel, row) in enumerate(ct.iterrows()):
    bottom = 0
    for val in row:
        if val > 0:
            ax.text(i, bottom + val / 2, int(val), ha="center", va="center",
                    fontsize=9, color="white", fontweight="bold")
        bottom += val
    ax.text(i, bottom + 0.7, int(row.sum()), ha="center", fontsize=9)

ax.set_ylim(0, 50)
ax.set_ylabel("domains")
ax.set_xlabel("DKIM selector")
ax.legend(["1024-bit (below RFC 8301 recommendation)",
           "2048-bit (meets recommendation)"], frameon=False, fontsize=9)
ax.set_title("DKIM key length by selector (n=72 confirmed records)")
plt.tight_layout()
plt.show()

In [ ]:
# key reuse: identical published public keys across unrelated domains
df["pk"] = df["dkim_record"].str.extract(r"(?:^|;)\s*p=([A-Za-z0-9+/=]+)")
shared = df["pk"].value_counts()[lambda s: s > 1].index

print("records:", df["pk"].notna().sum(), " distinct keys:", df["pk"].nunique())
print("domains sharing a key:", df["pk"].isin(shared).sum())
print()
for k in shared:
    grp = df.loc[df["pk"] == k]
    print(f"{len(grp)} domains | selector {grp['dkim_selector'].iloc[0]} "
          f"| {int(grp['dkim_key_bits'].iloc[0])}-bit")
    print("  ", ", ".join(grp["domain"]))

In [ ]:
counts("dkim_revoked")

In [ ]:
counts("dkim_testing_mode")

In [ ]:
df["dkim_strong"] = df["dkim_key_bits"] >= 2048
print("DKIM strong:", df["dkim_strong"].sum())

## 5.4 DMARC

In [ ]:
counts("dmarc_status")

In [ ]:
df.loc[df["dmarc_status"] != "found", ["organization_name", "domain", "sector_group"]]

In [ ]:
counts("dmarc_policy")

**Figure 5.6: DMARC policy distribution**

In [ ]:
POLICY_ORDER = ["reject", "quarantine", "none"]
p = df["dmarc_policy"].value_counts().reindex(POLICY_ORDER)

ax = p.plot.bar(color=[STRONG, PARTIAL, WEAK], rot=0)
label_bars(ax)
ax.set_ylim(0, 46)
ax.set_ylabel("domains")
ax.set_xlabel("organisational policy")
ax.set_title("DMARC policy")
plt.tight_layout()
plt.show()

In [ ]:
counts("dmarc_subdomain_policy")

**Figure 5.7: Organisational policy against subdomain policy**

In [ ]:
comp = pd.DataFrame({
    "organisational (p)": df["dmarc_policy"].value_counts().reindex(POLICY_ORDER),
    "subdomain (sp)": df["dmarc_subdomain_policy"].value_counts().reindex(POLICY_ORDER),
})

ax = comp.plot.bar(color=[PRIMARY, LIGHT], rot=0, figsize=(7, 4))
label_bars(ax)
ax.set_ylim(0, 46)
ax.set_ylabel("domains")
ax.legend(frameon=False)
ax.set_title("DMARC policy against subdomain policy")
plt.tight_layout()
plt.show()

In [ ]:
# sp is only meaningful where explicitly published; otherwise it inherits p
df["sp_explicit"] = df["dmarc_record"].str.contains("sp=", na=False)
print("explicit sp tag:", df["sp_explicit"].sum())
print()
print(pd.crosstab(df["dmarc_policy"], df["dmarc_subdomain_policy"]).to_string())
print()
gap = (df["dmarc_policy"].isin(["reject", "quarantine"])
       & (df["dmarc_subdomain_policy"] == "none"))
print("subdomain gap:", gap.sum())
print(df.loc[gap, ["domain", "dmarc_policy", "dmarc_subdomain_policy"]].to_string(index=False))

In [ ]:
counts("dmarc_alignment_spf")

In [ ]:
counts("dmarc_alignment_dkim")

In [ ]:
counts("dmarc_pct")

In [ ]:
# the single domain applying its policy to less than all mail
row = df.loc[df["dmarc_pct"] < 100].iloc[0]
print(row["organization_name"], "-", row["domain"])
print(row["dmarc_record"])

In [ ]:
counts("dmarc_reporting_configured")

In [ ]:
counts("dmarc_forensic_reporting_configured")

In [ ]:
no_rua = ~df["dmarc_reporting_configured"].astype(bool)
print(df.loc[no_rua, ["domain", "dmarc_policy"]].to_string(index=False))

In [ ]:
df["dmarc_enforcing"] = (df["dmarc_policy"].isin(["quarantine", "reject"])
                         & (df["dmarc_pct"] == 100))
print("DMARC enforcing:", df["dmarc_enforcing"].sum())

## 5.5 Combined effectiveness

**Table 5.1 / Figure 5.8: Presence against effective configuration**

In [ ]:
published = [78, 77, 72]
effective = [df["spf_effective"].sum(), df["dmarc_enforcing"].sum(), df["dkim_strong"].sum()]

t51 = pd.DataFrame({"published": published, "effective": effective},
                   index=["SPF", "DMARC", "DKIM"])
t51["difference_pp"] = ((t51["effective"] - t51["published"]) / N * 100).round(1)
t51

In [ ]:
ax = t51[["published", "effective"]].plot.bar(color=[LIGHT, PRIMARY], rot=0)
label_bars(ax)
ax.set_ylim(0, 90)
ax.set_ylabel("domains")
ax.legend(frameon=False)
ax.set_title("Presence against effective configuration")
plt.tight_layout()
plt.show()

In [ ]:
df["protected"] = df["spf_effective"] & df["dmarc_enforcing"] & df["dkim_strong"]
print(f"fully protected: {df['protected'].sum()} of {N} "
      f"({df['protected'].sum()/N*100:.1f}%)")

**Table 5.2: Protocol combinations in effect**

In [ ]:
t52 = (df.groupby(["spf_effective", "dmarc_enforcing", "dkim_strong"])
         .size().rename("domains")
         .reindex(pd.MultiIndex.from_product([[True, False]] * 3,
                  names=["spf_effective", "dmarc_enforcing", "dkim_strong"]),
                  fill_value=0)
         .reset_index())
t52 = t52.replace({True: "Yes", False: "No"})
t52

**Figure 5.9: Protocol combinations in effect**

In [ ]:
def pattern(r):
    return "/".join(["SPF" if r.spf_effective else "-",
                     "DMARC" if r.dmarc_enforcing else "-",
                     "DKIM" if r.dkim_strong else "-"])

df["pattern"] = df.apply(pattern, axis=1)
df["n_effective"] = df[["spf_effective", "dmarc_enforcing", "dkim_strong"]].sum(axis=1)

s = df.groupby("pattern")["n_effective"].first().to_frame("n")
s["count"] = df["pattern"].value_counts()
s = s.sort_values(["n", "count"])

shade = {0: WEAK, 1: PARTIAL, 2: LIGHT, 3: STRONG}
ax = plt.subplots(figsize=(8, 4))[1]
ax.barh(s.index, s["count"], color=[shade[n] for n in s["n"]])
for i, v in enumerate(s["count"]):
    ax.text(v + 0.5, i, f"{v}  ({v/N*100:.1f}%)", va="center", fontsize=9)
ax.set_xlim(0, 42)
ax.set_xlabel("domains")
ax.set_title("Protocol combinations in effect")
ax.legend(handles=[Patch(color=STRONG, label="all three effective"),
                   Patch(color=LIGHT, label="two effective"),
                   Patch(color=PARTIAL, label="one effective"),
                   Patch(color=WEAK, label="none effective")],
          frameon=False, fontsize=8.5, loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# composite on the thesis scale: SPF 1-4, DMARC 1-4, DKIM 1-3, maximum 11
# min_count=3 applies the "missing -> null, excluded from scoring" rule
df["composite"] = df[["spf_qualifier_score", "dmarc_policy_score",
                      "dkim_key_length_score"]].sum(axis=1, min_count=3)
counts("composite")

**Figure 5.10: Composite score distribution**

In [ ]:
c = df["composite"].value_counts().sort_index()
shade = [WEAK, WEAK, PARTIAL, LIGHT, STRONG]

ax = c.plot.bar(color=shade, rot=0)
label_bars(ax)
ax.set_ylim(0, 34)
ax.set_xlabel("composite score (maximum 11)")
ax.set_ylabel("domains")
ax.set_title(f"Composite score (n={int(c.sum())}, "
             f"{int(df['composite'].isna().sum())} excluded)")
plt.tight_layout()
plt.show()

**Table 5.3: Composite score for high-value state bodies**

In [ ]:
HIGH_VALUE = ["taoiseach.gov.ie", "finance.gov.ie", "justice.ie", "defence.ie",
              "dfa.ie", "centralbank.ie", "garda.ie", "hse.ie", "esb.ie",
              "eirgrid.com", "anpost.ie", "housing.gov.ie", "health.gov.ie",
              "education.gov.ie", "welfare.ie", "enterprise.gov.ie",
              "per.gov.ie", "transport.gov.ie"]

(df.loc[df["domain"].isin(HIGH_VALUE),
        ["organization_name", "domain", "composite",
         "spf_qualifier", "dmarc_policy", "dkim_key_bits"]]
   .sort_values("composite", na_position="first"))

## 5.6 Variation across the public sector

**Table 5.4 / Figure 5.11: DMARC enforcement by sector**

In [ ]:
t54 = pd.crosstab(df["sector_group"], df["dmarc_enforcing"]).reindex(SECTOR_ORDER)
t54.columns = ["not enforcing", "enforcing"]
t54["rate_pct"] = (t54["enforcing"] / t54.sum(axis=1) * 100).round(1)
t54

In [ ]:
rate = (t54["enforcing"] / (t54["enforcing"] + t54["not enforcing"]) * 100)

ax = rate.plot.bar(color=PRIMARY, rot=20)
label_bars(ax, fmt="{:.1f}%")
ax.set_ylim(0, 100)
ax.set_ylabel("% enforcing DMARC")
ax.set_xlabel("")
ax.set_title("DMARC enforcement by sector")
plt.tight_layout()
plt.show()

In [ ]:
# key length by sector, for comparison with the policy result above
kl = pd.crosstab(df["sector_group"], df["dkim_key_bits"]).reindex(SECTOR_ORDER)
(kl.div(kl.sum(axis=1), axis=0) * 100).round(1)

In [ ]:
# SPF lookup burden by sector
df.groupby("sector_group")["spf_lookup_count"].describe()[["count", "mean", "max"]].round(2)

In [ ]:
# Health reported at organisation level, not as proportions
df.loc[df["sector_group"] == "Health",
       ["organization_name", "domain", "spf_qualifier",
        "dmarc_policy", "dkim_key_bits", "protected"]]

## 5.7 Attack exposure

Each class is an observable precondition for T1684.002 (Email Spoofing).

**Table 5.5: Attack classes and exposure**

In [ ]:
df["dmarc_reporting_configured"] = df["dmarc_reporting_configured"].astype(bool)

attacks = {
    "Insufficient key margin":    (df["dkim_key_bits"] == 1024,            "Low now / strategic"),
    "Subdomain impersonation":    (df["dmarc_subdomain_policy"] == "none", "High"),
    "Direct domain spoofing":     (~df["dmarc_enforcing"],                 "High"),
    "Correlated key compromise":  (df["pk"].isin(shared),                  "Low now / high impact"),
    "Loss of DKIM fallback":      (df["dkim_status"] != "found",           "Medium"),
    "SPF-only bypass":            ((df["spf_qualifier"] == "?all")
                                   | df["spf_lookup_exceeds_limit"],       "Medium"),
    "Partial-enforcement bypass": (df["dmarc_pct"] < 100,                  "High"),
}

t55 = pd.DataFrame([{"attack_class": a, "domains": int(m.sum()),
                     "pct": round(m.sum() / N * 100, 1), "feasibility": f}
                    for a, (m, f) in attacks.items()]).sort_values("domains", ascending=False)
t55

**Figure 5.12: Attack classes by exposure and feasibility**

In [ ]:
FEAS = {"High": WEAK, "Medium": PARTIAL,
        "Low now / high impact": STRONG, "Low now / strategic": STRONG}

t = t55.sort_values("domains")
ax = plt.subplots(figsize=(8, 4))[1]
ax.barh(t["attack_class"], t["domains"], color=[FEAS[f] for f in t["feasibility"]])
for i, v in enumerate(t["domains"]):
    ax.text(v + 0.8, i, f"{v}  ({v/N*100:.1f}%)", va="center", fontsize=9)
ax.set_xlim(0, 66)
ax.set_xlabel("domains meeting precondition")
ax.set_title("Attack classes by exposure and feasibility (n=78)")
ax.legend(handles=[Patch(color=WEAK, label="high feasibility"),
                   Patch(color=PARTIAL, label="medium"),
                   Patch(color=STRONG, label="low now, high impact")],
          frameon=False, fontsize=8.5, loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# reported separately: a detection gap rather than an adversary technique
no_rua = ~df["dmarc_reporting_configured"]
print("no aggregate reporting:", no_rua.sum())
print("of which enforcing:", (no_rua & df["dmarc_enforcing"]).sum())
print(df.loc[no_rua & df["dmarc_enforcing"], ["domain", "dmarc_policy"]].to_string(index=False))

**Figure 5.13: Cumulative exposure per domain**

In [ ]:
A = pd.DataFrame({a: m for a, (m, _) in attacks.items()})
df["n_attacks"] = A.sum(axis=1)
df["attacks_exposed"] = A.apply(lambda r: "; ".join(A.columns[r]), axis=1)

c = df["n_attacks"].value_counts().sort_index()
shade = [STRONG if i == 0 else PARTIAL if i <= 2 else WEAK for i in c.index]

ax = plt.subplots()[1]
ax.bar(c.index, c.values, color=shade)
for i, v in zip(c.index, c.values):
    ax.text(i, v + 0.5, str(v), ha="center", fontsize=9)
ax.set_xlabel("attack classes exposed to")
ax.set_ylabel("domains")
ax.set_ylim(0, 36)
ax.set_title("Cumulative exposure per domain")
plt.tight_layout()
plt.show()

**Figure 5.14: Most exposed organisations**

In [ ]:
SECTOR_COLOUR = dict(zip(SECTOR_ORDER, [WEAK, PRIMARY, STRONG, PARTIAL, GREY]))
top = df.nlargest(15, "n_attacks").sort_values("n_attacks")

ax = plt.subplots(figsize=(8.5, 5))[1]
ax.barh(range(len(top)), top["n_attacks"],
        color=[SECTOR_COLOUR[s] for s in top["sector_group"]])
ax.set_yticks(range(len(top)))
ax.set_yticklabels([n[:46] for n in top["organization_name"]], fontsize=8)
ax.set_xlim(0, 6)
ax.set_xlabel("attack classes exposed to")
ax.set_title("Most exposed organisations")
ax.legend(handles=[Patch(color=v, label=k) for k, v in SECTOR_COLOUR.items()],
          frameon=False, fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# organisations meeting no precondition
df.loc[df["n_attacks"] == 0, ["organization_name", "sector_group"]]

**Figure 5.15: Attack exposure by sector**

In [ ]:
long = pd.concat([df.loc[m, ["sector_group"]].assign(attack=a)
                  for a, (m, _) in attacks.items()], ignore_index=True)
piv = pd.crosstab(long["attack"], long["sector_group"]).reindex(columns=SECTOR_ORDER,
                                                               fill_value=0)
rate = (piv / DENOM * 100).round(0)

fig, ax = plt.subplots(figsize=(7.5, 4))
im = ax.imshow(rate.values, cmap="Reds", vmin=0, vmax=100, aspect="auto")
ax.set_xticks(range(len(SECTOR_ORDER)))
ax.set_xticklabels([f"{o}\n(n={DENOM[o]})" for o in SECTOR_ORDER], fontsize=8)
ax.set_yticks(range(len(rate)))
ax.set_yticklabels(rate.index, fontsize=9)
for i in range(rate.shape[0]):
    for j in range(rate.shape[1]):
        v = rate.values[i, j]
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8,
                color="white" if v > 55 else "black")
ax.grid(False)
ax.set_title("% of sector meeting each attack precondition")
plt.colorbar(im, label="% of sector")
plt.tight_layout()
plt.show()

## Headline figures

Every number quoted in Chapter 5.

In [ ]:
print(f"Domains                        {N}")
print(f"SPF published / effective      {(df.spf_status=='found').sum()} / {df.spf_effective.sum()}")
print(f"DMARC published / enforcing    {(df.dmarc_status=='found').sum()} / {df.dmarc_enforcing.sum()}")
print(f"DKIM confirmed / 2048-bit      {(df.dkim_status=='found').sum()} / {df.dkim_strong.sum()}")
print("-" * 52)
print(f"FULLY PROTECTED                {df.protected.sum()} ({df.protected.sum()/N*100:.1f}%)")
print("-" * 52)
print(f"Subdomain gap (explicit)       {(df.dmarc_policy.isin(['reject','quarantine']) & (df.dmarc_subdomain_policy=='none')).sum()}")
print(f"Shared DKIM keys               {df.pk.isin(shared).sum()} domains, {len(shared)} keys")
print(f"1024-bit keys                  {(df.dkim_key_bits==1024).sum()} of {(df.dkim_status=='found').sum()} confirmed")
print(f"No aggregate reporting         {(~df.dmarc_reporting_configured).sum()}")
print(f"Exposed to no attack class     {(df.n_attacks==0).sum()}")